# Safebooru 메타데이터 크롤링
Safebooru API에서 메타데이터(태그 + URL)만 수집해 Parquet으로 저장합니다.
- **이미지는 저장하지 않음** — 학습 시 배치 단위로 임시 다운로드
- 저장 컬럼: `id`, `tags`, `file_url`, `sample_url`, `width`, `height`
- 중단 후 이어서 크롤링 가능

In [ ]:
import requests
import pandas as pd
import os
import time
from tqdm import tqdm
import xml.etree.ElementTree as ET

In [ ]:
# --- 경로 설정 ---
data_dir = '../data'
metadata_path = os.path.join(data_dir, 'metadata.parquet')

# --- API 설정 ---
API_URL = 'https://safebooru.org/index.php'
LIMIT = 1000        # API 최대 반환 수
DELAY = 0.5         # 요청 간 딜레이 (초)
MAX_RETRIES = 3     # 페이지당 최대 재시도
SAVE_INTERVAL = 10 # N 페이지마다 중간 저장
MAX_PAGES = None    # 크롤링할 최대 페이지 수 (None이면 전체, 예: 100 → 10만 건)

# --- 저장 컬럼 ---
KEEP_COLUMNS = ['id', 'tags', 'file_url', 'sample_url', 'width', 'height']

## 1. 전체 게시물 수 확인

In [ ]:
# XML 요청 1회로 총 건수 확인 (1초 이내)
params = {'page': 'dapi', 's': 'post', 'q': 'index', 'limit': 1}
response = requests.get(API_URL, params=params, timeout=30)
root = ET.fromstring(response.text)
total_count = int(root.attrib['count'])
total_pages = (total_count + LIMIT - 1) // LIMIT

if MAX_PAGES is not None:
    total_pages = min(total_pages, MAX_PAGES)

est_minutes = total_pages * DELAY / 60
print(f'전체 게시물 수: {total_count:,}')
print(f'크롤링 대상:    {total_pages:,} 페이지')
print(f'예상 소요 시간: 약 {est_minutes:.0f}분')

전체 게시물 수: 6,418,935
크롤링 대상:    6,419 페이지
예상 소요 시간: 약 53분


## 2. 메타데이터 크롤링

In [ ]:
# 이어서 크롤링 지원
start_pid = 0
existing_ids = set()
collected = 0

if os.path.exists(metadata_path):
    existing_df = pd.read_parquet(metadata_path)
    existing_ids = set(existing_df['id'].values)
    collected = len(existing_df)
    # 안전 마진: 2페이지 뒤로 (중복은 existing_ids로 걸러짐)
    start_pid = max(0, collected // LIMIT - 2)
    print(f'기존 데이터 {collected:,}건. pid={start_pid}부터 이어서 크롤링합니다.')
else:
    existing_df = None
    print('처음부터 크롤링을 시작합니다.')

buffer = []
done = False
new_count = 0

# 기존에 꼬여있는 tqdm 인스턴스 모두 제거
try:
    tqdm._instances.clear()
except:
    pass

pbar = tqdm(range(start_pid, total_pages), initial=start_pid, total=total_pages, desc='크롤링')

for pid in pbar:
    for attempt in range(MAX_RETRIES):
        try:
            params = {
                'page': 'dapi', 's': 'post', 'q': 'index',
                'limit': LIMIT, 'pid': pid, 'json': 1
            }
            resp = requests.get(API_URL, params=params, timeout=30)
            posts = resp.json()

            if not isinstance(posts, list) or not posts:
                done = True
                break

            for post in posts:
                if post['id'] not in existing_ids:
                    buffer.append({col: post.get(col) for col in KEEP_COLUMNS})
                    existing_ids.add(post['id'])
                    new_count += 1
            break

        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                time.sleep(DELAY * (attempt + 2))
            else:
                tqdm.write(f'pid={pid} 최종 실패: {e}')

    if done:
        break

    pbar.set_postfix({'수집': f'{collected + new_count:,}건', '신규': f'{new_count:,}건'})

    # 중간 저장
    if buffer and (pid + 1) % SAVE_INTERVAL == 0:
        new_df = pd.DataFrame(buffer)
        if existing_df is not None:
            new_df = pd.concat([existing_df, new_df], ignore_index=True)
        new_df = new_df.drop_duplicates(subset='id')
        new_df.to_parquet(metadata_path, index=False)
        existing_df = new_df
        buffer = []
        tqdm.write(f'중간 저장: {len(existing_df):,}건')

    time.sleep(DELAY)

# 최종 저장
if buffer:
    new_df = pd.DataFrame(buffer)
    if existing_df is not None:
        new_df = pd.concat([existing_df, new_df], ignore_index=True)
    new_df = new_df.drop_duplicates(subset='id')
    
    # --- 추가할 데이터 타입 정제 코드 시작 ---
    new_df['id'] = pd.to_numeric(new_df['id'], errors='coerce')
    new_df['width'] = pd.to_numeric(new_df['width'], errors='coerce')
    new_df['height'] = pd.to_numeric(new_df['height'], errors='coerce')

    # 문자열 컬럼의 경우 빈 값(None)을 빈 문자열로 일괄 치환하여 타입 충돌 원천 차단
    new_df['tags'] = new_df['tags'].fillna('').astype(str)
    new_df['file_url'] = new_df['file_url'].fillna('').astype(str)
    new_df['sample_url'] = new_df['sample_url'].fillna('').astype(str)
    # --- 추가할 데이터 타입 정제 코드 끝 ---

    new_df.to_parquet(metadata_path, index=False)

df = pd.read_parquet(metadata_path)
file_size_mb = os.path.getsize(metadata_path) / (1024 * 1024)
print(f'\n크롤링 완료: {len(df):,}건 (신규 {new_count:,}건)')
print(f'파일 크기: {file_size_mb:.1f} MB')
print(f'저장 위치: {os.path.abspath(metadata_path)}')

처음부터 크롤링을 시작합니다.


크롤링:   0%|          | 9/6419 [00:37<7:21:43,  4.13s/it, 수집=10,000건, 신규=10,000건]


ArrowKeyError: No type extension with name arrow.py_extension_type found

## 3. 확인

In [ ]:
df = pd.read_parquet(metadata_path)
print(f'총 {len(df):,}건')
print(f'컬럼: {list(df.columns)}')
print(f'URL 없는 행: {df["sample_url"].isna().sum():,}건')
print(f'태그 없는 행: {df["tags"].isna().sum():,}건')
df.head(3)